In [ ]:
from ase.build import bulk
from executorlib import FluxClusterExecutor
from jinja2 import Template
from lammpsparser import get_potential_dataframe
from pylammpsmpi import LammpsASELibrary, init_function

In [ ]:
structure = bulk("Au", cubic=True).repeat((5,5,5))
element_lst = structure.get_chemical_symbols()
element_lst[0] = "Cu"
structure.set_chemical_symbols(element_lst)
structure

In [ ]:
df_pot = get_potential_dataframe(structure)
df_pot

In [ ]:
potential = "1990--Ackland-G-J--Cu-Ag-Au--LAMMPS--ipr1"
df_pot[df_pot["Name"] == potential]["Species"].values[0], df_pot[df_pot["Name"] == potential]["Config"].values[0]

In [ ]:
LAMMPS_template = """\
thermo {{thermo}}
thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep {{timestep}}
velocity all create $({{velocity_rescale_factor}} * {{ temp }}) {{seed}} dist {{dist}}
fix ensemble all nvt temp {{Tstart}} {{Tstop}} {{Tdamp}}
"""

In [ ]:
template = Template(LAMMPS_template)
lmp_str = template.render(
    thermo=100,
    timestep=0.001,
    velocity_rescale_factor=2.0,
    temp=300.0,
    seed=12345,
    dist="gaussian",
    Tstart=300.0,
    Tstop=300.0,
    Tdamp=0.1,
)
lmp_str

In [ ]:
with FluxClusterExecutor( 
    max_workers=1, 
    block_allocation=True, 
    init_function=init_function,
    resource_dict={
        # "submission_template": submission_template, 
        # "run_time_max": 180,  # in seconds  
        # "partition": "s.cmfe",
        "cores": 2,
        "threads_per_core": 1,
    },
) as exe:
    lmp = LammpsASELibrary(executor=exe, cores=2)
    lmp.interactive_structure_setter(
        structure=structure,
        units="metal",
        dimension=3,
        boundary=" ".join(["p" if coord else "f" for coord in structure.pbc]),
        atom_style="atomic",
        el_eam_lst=df_pot[df_pot["Name"] == potential]["Species"].values[0],
        calc_md=True,
    )
    for c in df_pot[df_pot["Name"] == potential]["Config"].values[0]:
        lmp.interactive_lib_command(c)
    if lmp_str is not None:
        for line in lmp_str.split("\n"):
            lmp.interactive_lib_command(line)

    lmp.interactive_lib_command("run 100")
    print(lmp.interactive_energy_pot_getter())
    lmp.close()